# 🧪 Clase 5 — Evaluación de modelos y supuestos estadísticos
## Unidad: Correlación y modelamiento

**Situación:** La gerencia quiere predecir las ventas mensuales a partir de visitas, publicidad y cantidad de vendedores. Tras ajustar el modelo, te pide verificar si es **explicativo** y **estadísticamente válido**.

**Objetivos:**
- Entender y calcular **R² vs R² ajustado**
- Aplicar **tests de normalidad** de residuos: Shapiro-Wilk, Jarque-Bera, Kolmogorov-Smirnov
- Leer los tests de normalidad que ya incluye `modelo.summary()`
- Implementar el **flujo completo** con statsmodels
- Reconocer cuándo los supuestos se violan y qué hacer

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

print('✅ Librerías cargadas')
print(f'statsmodels {sm.__version__} | scipy {stats.__version__ if hasattr(stats, "__version__") else "OK"}')

---
## PARTE 1 — R² vs R² ajustado

### 1.1 El problema de R² sin penalización

In [ ]:
# Demostración: añadir una variable irrelevante sube R² pero baja R² ajustado
np.random.seed(42)
n_demo = 50

x1   = np.random.rand(n_demo) * 10
x2   = np.random.rand(n_demo) * 5     # segunda variable útil
ruido_irrel = np.random.rand(n_demo)  # variable irrelevante (puro ruido)
y_demo = 3 * x1 + 1.5 * x2 + np.random.normal(0, 2, n_demo)

# Modelo 1: solo x1
m1 = sm.OLS(y_demo, sm.add_constant(x1)).fit()
# Modelo 2: x1 + x2 (útil)
m2 = sm.OLS(y_demo, sm.add_constant(np.column_stack([x1, x2]))).fit()
# Modelo 3: x1 + x2 + variable irrelevante
m3 = sm.OLS(y_demo, sm.add_constant(np.column_stack([x1, x2, ruido_irrel]))).fit()

tabla_r2 = pd.DataFrame({
    'Modelo':        ['Modelo 1 (1 var)', 'Modelo 2 (2 vars)', 'Modelo 3 (+ variable irrelevante)'],
    'N predictores': [1, 2, 3],
    'R²':            [m1.rsquared,     m2.rsquared,     m3.rsquared],
    'R² ajustado':   [m1.rsquared_adj, m2.rsquared_adj, m3.rsquared_adj],
    'AIC':           [m1.aic,          m2.aic,          m3.aic],
}).round(5)

print('=== Efecto de añadir variables al modelo ===')
print(tabla_r2.to_string(index=False))
print()
print('Observación clave:')
print(f'  R² subió de {m2.rsquared:.4f} → {m3.rsquared:.4f} al añadir la variable irrelevante')
print(f'  R² ajustado BAJÓ de {m2.rsquared_adj:.4f} → {m3.rsquared_adj:.4f} ← señal de que la variable no aporta')

In [ ]:
# Tabla comparativa de la presentación
tabla_pres = pd.DataFrame({
    'Modelo':         ['Modelo 1', 'Modelo 2 (+ variable irrelevante)'],
    'Nº predictores': [2, 3],
    'R²':             [0.85, 0.86],
    'R² ajustado':    [0.83, 0.82]
})
print('=== Tabla comparativa de la presentación ===')
print(tabla_pres.to_string(index=False))
print()
print('Conclusión: aunque R² subió, R² ajustado bajó → la variable extra no aportó valor real')

### 1.2 Cálculo con statsmodels — flujo de la presentación

In [ ]:
# Código exacto de la presentación (slides 8-9)
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 1. Crear datos de ejemplo
np.random.seed(42)
X_ej = np.random.rand(50) * 10
Y_ej = 2 * X_ej + np.random.normal(0, 1.5, 50)

# 2. DataFrame
data = pd.DataFrame({'X': X_ej, 'Y': Y_ej})

# 3. Definir variables
Y_modelo = data['Y']
X_modelo = data['X']

# 4. Agregar constante
X_modelo = sm.add_constant(X_modelo)

# 5. Crear y ajustar el modelo OLS
modelo = sm.OLS(Y_modelo, X_modelo)
resultados = modelo.fit()

# 6. Resumen
print(resultados.summary())

In [ ]:
# Leer los R² directamente
r2     = resultados.rsquared
r2_adj = resultados.rsquared_adj
n_obs  = int(resultados.nobs)
p_pred = resultados.df_model  # número de predictores

print('=== Indicadores de ajuste ===')
print(f'R²:          {r2:.6f}')
print(f'R² ajustado: {r2_adj:.6f}')
print(f'N obs:       {n_obs}')
print(f'N predictores: {int(p_pred)}')
print()
print('Fórmula manual R² adj:')
r2_adj_manual = 1 - (1 - r2) * (n_obs - 1) / (n_obs - p_pred - 1)
print(f'  1 - (1 - {r2:.4f}) × ({n_obs}-1) / ({n_obs}-{int(p_pred)}-1) = {r2_adj_manual:.6f} ✅')

### ✏️ Ejercicio 1:

In [ ]:
# ✏️ ¿Cuándo R² y R² ajustado serán muy similares?
r_similar = ""

# ✏️ ¿Cuándo preferirías R² ajustado por encima de R²?
r_preferir_adj = ""

# ✏️ Si R² ajustado baja al añadir una variable, ¿qué deberías hacer?
r_baja = ""

print(f'R² ≈ R² adj cuando: {r_similar}')
print(f'Preferir R² adj:    {r_preferir_adj}')
print(f'Si R² adj baja:     {r_baja}')

---
## PARTE 2 — Test de normalidad de los residuos

### 2.1 Métodos visuales

In [ ]:
# Código exacto de la presentación
import matplotlib.pyplot as plt
import scipy.stats as stats_sp
import statsmodels.api as sm

residuos = resultados.resid

# Histograma — exacto de la presentación
plt.hist(residuos, bins=30, edgecolor='black')
plt.title('Histograma de residuos')
plt.show()

# Q-Q Plot — exacto de la presentación
sm.qqplot(residuos, line='s')
plt.title('Q-Q plot de residuos')
plt.show()

### 2.2 Tests estadísticos de normalidad

In [ ]:
# Código exacto de la presentación — Shapiro-Wilk
stat, p = stats_sp.shapiro(residuos)
print(f'Estadístico: {stat:.4f}, p-valor: {p:.4f}')
print()

# Interpretación exacta de la presentación
if p < 0.05:
    print('Resultado: p < 0.05 → Se rechaza H₀ (normalidad)')
    print('Los residuos probablemente NO siguen una distribución normal')
else:
    print('Resultado: p ≥ 0.05 → No se rechaza H₀ (normalidad)')
    print('Los residuos probablemente siguen una distribución normal')

In [ ]:
# Tabla completa de todos los tests — presentación slide 14
print('=== Batería completa de tests de normalidad ===')

# Shapiro-Wilk (recomendado < 5000 obs)
sw_stat, sw_p = stats_sp.shapiro(residuos)
print(f'Shapiro-Wilk:         W={sw_stat:.4f},  p={sw_p:.4f}')

# Kolmogorov-Smirnov (contra distribución normal)
res_norm = (residuos - residuos.mean()) / residuos.std()
ks_stat, ks_p = stats_sp.kstest(res_norm, 'norm')
print(f'Kolmogorov-Smirnov:   D={ks_stat:.4f},  p={ks_p:.4f}')

# Jarque-Bera (basado en sesgo y curtosis — viene en summary())
jb_stat, jb_p, skew, kurt = sm.stats.stattools.jarque_bera(residuos)
print(f'Jarque-Bera:          JB={jb_stat:.4f}, p={jb_p:.4f}')
print(f'  Sesgo (skew):    {skew:.4f}   |  Curtosis: {kurt:.4f}')

print()
tabla_tests = pd.DataFrame({
    'Test':         ['Shapiro-Wilk', 'Kolmogorov-Smirnov', 'Jarque-Bera'],
    'Tipo':         ['Estadístico','Estadístico','Estadístico'],
    'Descripción':  ['Recomendado n<5000','Compara con normal teórica','Basado en sesgo y curtosis'],
    'Estadístico':  [round(sw_stat,4), round(ks_stat,4), round(jb_stat,4)],
    'p-value':      [round(sw_p,4),    round(ks_p,4),    round(jb_p,4)],
    'Normal?':      ['✅' if sw_p>=0.05 else '⚠️',
                     '✅' if ks_p>=0.05 else '⚠️',
                     '✅' if jb_p>=0.05 else '⚠️']
})
print(tabla_tests.to_string(index=False))

In [ ]:
# Los tests de normalidad YA aparecen en summary() — cómo leerlos
print('=== Tests de normalidad en modelo.summary() ===')
print()
print('Dentro de summary(), la tabla inferior incluye:')
print('  Omnibus:       test basado en sesgo y curtosis')
print('  Prob(Omnibus): p-value → si < 0.05: no normal')
print('  Jarque-Bera:   otro test de normalidad')
print('  Prob(JB):      p-value → si < 0.05: no normal')
print('  Skew:          sesgo (idealmente ≈ 0)')
print('  Kurtosis:      curtosis (idealmente ≈ 3)')
print()
# Extraer JB del summary — código de la presentación
jb_test_p_value = resultados.summary().tables[2].data[1][3]
try:
    p_value_jb = float(jb_test_p_value)
except ValueError:
    _, p_value_jb = stats_sp.shapiro(resultados.resid)

print(f'📊 Valor P (Test de Normalidad de Residuos): {p_value_jb:.4f}')

# Interpretación exacta de la presentación
if p_value_jb < 0.05:
    print('🛑 ¡ALERTA! El p-valor es menor que 0.05. Se rechaza la H0.')
    print('Conclusión: Los residuos probablemente NO siguen una distribución normal.')
else:
    print('✅ El p-valor es mayor o igual que 0.05. No se rechaza la H0.')
    print('Conclusión: Los residuos probablemente siguen una distribución normal.')

### ✏️ Ejercicio 2:

In [ ]:
# ✏️ ¿Por qué siempre combinar test estadístico con visualización?
r_combinar = ""

# ✏️ Si p > 0.05 en Shapiro-Wilk, ¿podemos afirmar que los residuos SON normales?
r_confirmar = ""

# ✏️ ¿Qué pasaría con las inferencias (p-values, IC) si los residuos no son normales?
r_consecuencia = ""

print(f'Combinar: {r_combinar}')
print(f'p>0.05 confirma: {r_confirmar}')
print(f'Consecuencia: {r_consecuencia}')

---
## PARTE 3 — Flujo completo con statsmodels

### 3.1 Código exacto de la presentación (slides 20-21)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Generación de datos con NumPy
X_flujo = np.arange(1, 11)                              # Variable independiente
y_flujo = 2 * X_flujo + 5 + np.random.normal(0, 1, 10) # y = 2x + 5 + ruido

# Creación del DataFrame
data_flujo = pd.DataFrame({'X': X_flujo, 'y': y_flujo})

# Agregar la constante
X_con_constante = sm.add_constant(data_flujo['X'])

# Definir y ajustar el modelo OLS
modelo_flujo = sm.OLS(data_flujo['y'], X_con_constante)
resultados_flujo = modelo_flujo.fit()

# Imprimir resumen
print(resultados_flujo.summary())

In [ ]:
# Guía de lectura de la tabla de elementos — slide 21
print('=== Guía de lectura del summary() ===')
elementos = [
    ('R-squared / Adj. R²', 'Bondad de ajuste y penalización por complejidad'),
    ('coef / std err',       'Coeficientes e incertidumbre de cada variable'),
    ('t / P>|t|',            'Prueba de significancia de cada predictor'),
    ('Omnibus / Jarque-Bera','Test de normalidad de residuos'),
    ('Durbin-Watson',        'Test de autocorrelación de residuos (ideal ≈ 2)'),
    ('Cond. No.',            'Posible multicolinealidad (alerta si > 1000)'),
]
for indicador, significado in elementos:
    print(f'  {indicador:<30} → {significado}')

---
## PARTE 4 — Actividad guiada: Modelo de precios de vivienda

### 4.1 Preparación e inicialización (código exacto presentación)

In [ ]:
# Slide 25 — importar librerías
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

### 4.2 Generación y modelado de datos (código exacto presentación)

In [ ]:
# Slide 26 — Paso 2.1: Generar datos simulados
np.random.seed(42)
area_sqm     = np.random.normal(loc=150, scale=30, size=50)
precio_venta = 1000 * area_sqm + np.random.normal(0, 30000, 50)

data_inmobiliaria = pd.DataFrame({'Area_sqm': area_sqm, 'Precio': precio_venta})
print(data_inmobiliaria.describe().round(2))

In [ ]:
# Slide 27 — Paso 2.2: Ajustar el modelo OLS
X_inm = sm.add_constant(data_inmobiliaria['Area_sqm'])
y_inm = data_inmobiliaria['Precio']

modelo_ols = sm.OLS(y_inm, X_inm).fit()

### 4.3 Evaluación: R² ajustado (código exacto presentación)

In [ ]:
# Slide 28 — Paso 3.1: Obtener R² ajustado
r_squared_adj = modelo_ols.rsquared_adj
print(f'✅ R-cuadrado Ajustado: {r_squared_adj:.4f}')
print()
print(f'R² simple:   {modelo_ols.rsquared:.4f}')
print(f'R² ajustado: {modelo_ols.rsquared_adj:.4f}')
print()
print(f'Interpretación: el modelo explica el {r_squared_adj*100:.1f}% de la variabilidad del precio')

### 4.4 Test de normalidad de residuos (código exacto presentación)

In [ ]:
# Slide 29 — Paso 4.1: Extraer residuos y aplicar test
jb_test_p_value = modelo_ols.summary().tables[2].data[1][3]

try:
    p_value_jb = float(jb_test_p_value)
except ValueError:
    _, p_value_jb = stats.shapiro(modelo_ols.resid)

print(f'📊 Valor P (Test de Normalidad de Residuos): {p_value_jb:.4f}')

In [ ]:
# Slide 30 — Paso 4.2: Interpretación
if p_value_jb < 0.05:
    print('🛑 ¡ALERTA! El p-valor es menor que 0.05. Se rechaza la H0.')
    print('Conclusión: Los residuos probablemente NO siguen una distribución normal.')
    print('El modelo podría ser inválido para inferencia.')
else:
    print('✅ El p-valor es mayor o igual que 0.05. No se rechaza la H0.')
    print('Conclusión: Los residuos probablemente siguen una distribución normal.')
    print('El modelo es válido para inferencia.')

In [ ]:
# Slide 31 — Resumen completo
print('\n--- Resumen Completo del Modelo OLS ---')
print(modelo_ols.summary())

In [ ]:
# Diagnóstico visual completo
res_inm  = modelo_ols.resid
pred_inm = modelo_ols.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Diagnóstico completo — Modelo Precios de Vivienda', fontweight='bold', fontsize=12)

# 1. Scatter + recta
axes[0,0].scatter(data_inmobiliaria['Area_sqm'], y_inm,
                  color='#2E75B6', alpha=0.6, s=40)
x_r = np.linspace(area_sqm.min(), area_sqm.max(), 100)
b0_inm = modelo_ols.params['const']
b1_inm = modelo_ols.params['Area_sqm']
axes[0,0].plot(x_r, b0_inm + b1_inm*x_r, color='#ED7D31', linewidth=2.5)
axes[0,0].set_title(f'Recta de regresión | R²adj={r_squared_adj:.3f}')
axes[0,0].set_xlabel('Área (m²)')
axes[0,0].set_ylabel('Precio ($)')

# 2. Residuos vs predichos
axes[0,1].scatter(pred_inm, res_inm, color='#7030A0', alpha=0.6, s=35)
axes[0,1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0,1].set_title('Residuos vs Valores ajustados')
axes[0,1].set_xlabel('ŷ')
axes[0,1].set_ylabel('Residuo')

# 3. Q-Q plot
sm.qqplot(res_inm, line='s', ax=axes[1,0])
axes[1,0].set_title('Q-Q plot de residuos')

# 4. Histograma residuos
axes[1,1].hist(res_inm, bins=15, edgecolor='black', color='#BDD7EE')
axes[1,1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1,1].set_title('Histograma de residuos')
axes[1,1].set_xlabel('Residuo')

plt.tight_layout()
plt.show()

# Batería de tests
sw_s, sw_p2 = stats.shapiro(res_inm)
jb_s2, jb_p2, sk2, ku2 = sm.stats.stattools.jarque_bera(res_inm)
print(f'Shapiro-Wilk:  p={sw_p2:.4f}  → {"✅ Normal" if sw_p2>=0.05 else "⚠️ No normal"}')
print(f'Jarque-Bera:   p={jb_p2:.4f}  → {"✅ Normal" if jb_p2>=0.05 else "⚠️ No normal"}')
print(f'Sesgo (skew):  {sk2:.4f}       (ideal ≈ 0)')
print(f'Curtosis:      {ku2:.4f}       (ideal ≈ 3)')

---
## PARTE 5 — Aplicación: Modelo de ventas (desafío inicial)

### 5.1 Cargar y ajustar el modelo con datos reales

In [ ]:
df_v = pd.read_csv('ventas_prediccion.csv')
print(f'Registros: {len(df_v)}')
print(df_v.head(6))
print()
print(df_v.describe().round(2))

In [ ]:
# Modelo completo: visitas + publicidad + vendedores → ventas
X_v = df_v[['visitas_semana', 'inversion_publicidad', 'n_vendedores']]
X_v = sm.add_constant(X_v)
Y_v = df_v['ventas_mensuales']

modelo_v = sm.OLS(Y_v, X_v).fit()
print(modelo_v.summary())

In [ ]:
# Evaluación completa
res_v  = modelo_v.resid
r2_v   = modelo_v.rsquared
r2a_v  = modelo_v.rsquared_adj

# Tests de normalidad
sw_sv, sw_pv = stats.shapiro(res_v)
jb_sv, jb_pv, skv, kuv = sm.stats.stattools.jarque_bera(res_v)

print('=== Evaluación del modelo de ventas ===')
print(f'R²:            {r2_v:.4f}')
print(f'R² ajustado:   {r2a_v:.4f}')
print(f'AIC:           {modelo_v.aic:.2f}')
print()
print('Tests de normalidad de residuos:')
print(f'  Shapiro-Wilk: p={sw_pv:.4f}  → {"✅ Normal" if sw_pv>=0.05 else "⚠️ No normal"}')
print(f'  Jarque-Bera:  p={jb_pv:.4f}  → {"✅ Normal" if jb_pv>=0.05 else "⚠️ No normal"}')
print(f'  Sesgo:   {skv:.4f}  |  Curtosis: {kuv:.4f}')

# Responder al desafío
print()
print('=== Respuesta a la gerencia ===')
print(f'El modelo explica el {r2a_v*100:.1f}% de la variabilidad de las ventas (R² adj).')
sig = all(modelo_v.pvalues.drop('const') < 0.05)
print(f'Todos los predictores son significativos: {"✅ Sí" if sig else "⚠️ No"}')
norm = sw_pv >= 0.05
print(f'Residuos normales (Shapiro-Wilk): {"✅ Sí" if norm else "⚠️ Revisar"}')
print(f'Conclusión: el modelo {"es estadísticamente válido" if norm else "requiere revisión de supuestos"}')

### ✏️ Reflexiones finales:

In [ ]:
# ✏️ 1. ¿En qué casos podrías ver residuos no normales? ¿Qué soluciones existen?
c1 = ""

# ✏️ 2. Si Shapiro-Wilk da p < 0.05, ¿el modelo es inservible?
c2 = ""

# ✏️ 3. ¿Qué diferencia existe entre statsmodels y scikit-learn para regresión?
c3 = ""

# ✏️ 4. ¿Cómo comunicarías los resultados de este modelo a la gerencia?
c4 = ""

# ✏️ 5. ¿Cuáles son las 3 métricas que nunca deberían faltar en un reporte de modelo?
c5 = ""

print('--- REFLEXIONES FINALES ---')
for i, c in enumerate([c1, c2, c3, c4, c5], 1):
    print(f'{i}. {c}')

---
## 📋 Resumen: R² ajustado y tests de normalidad

### R² vs R² ajustado

| Indicador | Fórmula | Cuándo sube | Cuándo baja | Cuándo usar |
|-----------|---------|------------|------------|-------------|
| R² | 1 - SSE/SST | Siempre al añadir variables | Nunca | Regresión simple |
| R² ajustado | 1 - (1-R²)(n-1)/(n-p-1) | Solo si la var. aporta | Si la var. es irrelevante | Regresión múltiple |

### Tests de normalidad de residuos

| Test | Función | N recomendado | H₀ |
|------|---------|--------------|----|
| Shapiro-Wilk | `stats.shapiro(res)` | < 5000 | Residuos son normales |
| Kolmogorov-Smirnov | `stats.kstest(res,'norm')` | Cualquier N | Ajusta a normal |
| Jarque-Bera | `sm.stats.stattools.jarque_bera(res)` | Grande | Sesgo=0, Kurtosis=3 |
| Omnibus | En `modelo.summary()` | — | Normalidad de residuos |

**Regla de decisión:**
- `p > 0.05` → No se rechaza H₀ → Residuos compatibles con normalidad ✅  
- `p < 0.05` → Se rechaza H₀ → Evidencia de no normalidad ⚠️

> 💡 **Leer el `summary()` completo:** la tabla inferior ya incluye Omnibus, Jarque-Bera, Skew, Kurtosis y Durbin-Watson. No necesitas calcularlos por separado en una primera revisión.

> 💡 **statsmodels vs scikit-learn:** statsmodels = inferencia estadística (p-values, IC, supuestos). scikit-learn = predicción y Machine Learning (cross-validation, pipelines). Para regresión clásica con interpretación, statsmodels es la herramienta correcta.